# ACT run D, fase 1 — solo i 500 simulati

La rete impara la struttura del compito dove i dati costano poco. 60000 passi,
checkpoint ogni 20000. Poi `act-caramelle-run-d2` riparte da qui coi veri
dentro. Provata da sola sul braccio dice quanto la simulazione e' vicina al vero:
non ha mai visto un'immagine vera.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv; nproc; df -h /tmp | tail -1

In [ ]:
import glob, os, shutil
s = glob.glob('/kaggle/input/**/robotica/scripts/genera_act_sim.py', recursive=True)
assert s, 'pacchetto codice non trovato: e\' in Input?'
shutil.copytree(os.path.dirname(os.path.dirname(s[0])), '/tmp/robotica', dirs_exist_ok=True)
print('codice:', s[0])

In [ ]:
!pip install -q uv
!UV_CACHE_DIR=/tmp/uvcache uv venv -q -p 3.12 /tmp/venv
!UV_CACHE_DIR=/tmp/uvcache uv pip install -q --python /tmp/venv/bin/python --torch-backend=auto "lerobot[training,dataset]==0.6.1" "mujoco==3.3.7" scipy
!/tmp/venv/bin/python -c "import lerobot, mujoco, numpy; print('lerobot', lerobot.__version__, '| mujoco', mujoco.__version__, '| numpy', numpy.__version__)"

In [ ]:
import json, subprocess
sim = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/caramelle_sim/meta/info.json', recursive=True)]
assert len(sim) == 1, sim
es = json.load(open(sim[0] + '/meta/info.json'))['total_episodes']
SIM = sim[0]
print('simulati:', SIM, es, 'episodi')
if es != 500:
    print(f'ATTENZIONE: {es} simulati invece di 500')

In [ ]:
shutil.copytree(sim[0], '/tmp/dati', copy_function=shutil.copyfile)
info = json.load(open('/tmp/dati/meta/info.json'))
print('solo simulati: episodi', info['total_episodes'], 'fotogrammi', info['total_frames'])
REPO = 'so101/caramelle_sim'

In [ ]:
!/tmp/venv/bin/lerobot-train \
    --dataset.repo_id=$REPO \
    --dataset.root=/tmp/dati \
    --dataset.image_transforms.enable=true \
    --policy.type=act \
    --policy.device=cuda \
    --policy.push_to_hub=false \
    --output_dir=/kaggle/working/act_caramelle \
    --job_name=act_run_d1 \
    --steps=60000 \
    --save_freq=20000 \
    --batch_size=8 \
    --wandb.enable=false

In [ ]:
base = '/kaggle/working/act_caramelle/checkpoints'
numeri = {int(n): n for n in os.listdir(base) if n.isdigit()}
print('checkpoint:', sorted(numeri))
for passi in sorted(numeri):
    uscita = shutil.make_archive(f'/kaggle/working/modello_act_{passi}', 'zip', os.path.join(base, numeri[passi], 'pretrained_model'))
    print('pronto:', uscita, os.path.getsize(uscita) / 1e6, 'MB')